# Inheritance: Families of Wordle Players

Last time every player was its own class: `RandomPlayer`, `FilterPlayer`, and
`LLMPlayer`.  They all had a `name`, a `new_game` method, and a `next_guess`
method, and we wrote that shared code again in each class.  Today we put the
shared code in one place and build new players from old ones.

Run each cell, but try to **predict the output before you run it**.

## Setup

The same word lists, `score`, and `play` as last time.

In [17]:
import random
import urllib.request

URL = "https://raw.githubusercontent.com/hcai-programming/hcai121-fall2026/main/week10/"

def load_words(filename):
    with urllib.request.urlopen(URL + filename) as f:
        return f.read().decode().split()

answers = load_words("answers.txt")         # words that can be the secret word
guesses = load_words("valid_guesses.txt")   # every legal guess
legal = set(guesses)
print(len(answers), "possible answers,", len(guesses), "legal guesses")

2315 possible answers, 12972 legal guesses


In [18]:
def score(answer, guess):
    """Return a list of (letter, status) pairs, where status is green, yellow, or gray."""
    status = ["gray"] * len(guess)
    remaining = {}
    for i, (g, a) in enumerate(zip(guess, answer)):
        if g == a:
            status[i] = "green"
        else:
            remaining[a] = remaining.get(a, 0) + 1
    for i, (g, a) in enumerate(zip(guess, answer)):
        if status[i] == "gray" and remaining.get(g, 0) > 0:
            status[i] = "yellow"
            remaining[g] = remaining[g] - 1
    return list(zip(guess, status))

def play(answer, player, max_turns=6):
    player.new_game()
    history = []
    for turn in range(1, max_turns + 1):
        guess = player.next_guess(history)
        feedback = score(answer, guess)
        history.append((guess, feedback))
        if guess == answer:
            return turn
    return None

## 1. A base class

A **base class** (also called a *parent class*) holds what every player has in
common.  Here a basic `Player` has a name, has nothing to reset at the start
of a game, and guesses a random legal word.

In [19]:
class Player:
    def __init__(self, name):
        self.name = name

    def new_game(self):
        pass

    def next_guess(self, history):
        return random.choice(guesses)

## 2. A subclass inherits everything

A **subclass** (also called a *child class*) is written with the parent class
in parentheses.  Without adding a single line, it **inherits** every method of
the parent, including `__init__`.

In [20]:
class RandomPlayer(Player):
    pass

r = RandomPlayer("random")
print(r.name)
print(play("crane", r))

random
None


`RandomPlayer` has no `__init__` of its own, so `RandomPlayer("random")` runs
`Player.__init__`.  A `RandomPlayer` *is a* `Player`:

In [22]:
print(isinstance(r, RandomPlayer))
print(isinstance(r, Player))


True
True


## 3. A subclass can override methods

A subclass can **override** a method: if it defines a method with the same
name, its version is used instead of the parent's.  `FilterPlayer` keeps
`Player.__init__` but replaces `new_game` and `next_guess`.

In [24]:
class FilterPlayer(Player):
    def new_game(self):
        self.candidates = list(answers)

    def next_guess(self, history):
        if history:
            last_guess, last_feedback = history[-1]
            self.candidates = [w for w in self.candidates
                               if score(w, last_guess) == last_feedback]
        return self.candidates[0]

f = FilterPlayer("filter")
print(f.name)
print(play("crane", f))

filter
4


When you call `f.next_guess(history)`, Python looks for `next_guess` in
`FilterPlayer` first.  It finds it there, so it never looks in `Player`.  When
you ask for `f.name`, the value was set by `Player.__init__`, because
`FilterPlayer` has no `__init__` of its own.

## 4. Building on the parent: `super()`

Many Wordle players always open with the same word.  We want a player that
works exactly like `FilterPlayer`, except that its first guess is a word we
choose.  We should not copy `FilterPlayer`: we subclass it and change only
what is different.

`super()` means *the parent class's version*.  `super().__init__(name)` runs
the parent's `__init__`, and `super().next_guess(history)` runs the parent's
`next_guess`.

In [28]:
class StartWordPlayer(FilterPlayer):
    def __init__(self, name, start_word):
        super().__init__(name)          # set self.name, as Player does
        self.start_word = start_word    # the one new attribute

    def next_guess(self, history):
        if not history:
            return self.start_word
        return super().next_guess(history)

s = StartWordPlayer("slate opener", "slate")
print(s.name, s.start_word)
print(play("crane", s))

slate opener slate
4


`StartWordPlayer` has a parent (`FilterPlayer`) and a grandparent (`Player`),
and it is an instance of all three:

In [29]:
print(isinstance(s, StartWordPlayer), isinstance(s, FilterPlayer), isinstance(s, Player))

True True True


What happens if a subclass writes its own `__init__` and forgets to call
`super().__init__`?

In [30]:
class ForgetfulPlayer(FilterPlayer):
    def __init__(self, name, start_word):
        self.start_word = start_word

p = ForgetfulPlayer("forgetful", "slate")
print(p.start_word)
print(p.name)

slate


AttributeError: 'ForgetfulPlayer' object has no attribute 'name'

`ForgetfulPlayer.__init__` replaces `Player.__init__`, so the line
`self.name = name` never runs and the object has no `name` attribute.  If you
override `__init__`, call `super().__init__(...)` first.

## 5. Does a good opening word help?

Every player is a `Player`, so we can put them in one list and test them on
the same secret words.  A lost game counts as 7 turns.  **Lower is better.**

In [31]:
random.seed(1)
secret_words = random.sample(answers, 100)

def average_turns(player, secret_words):
    total = 0
    for answer in secret_words:
        turns = play(answer, player)
        if turns is None:
            turns = 7
        total = total + turns
    return total / len(secret_words)

players = [RandomPlayer("random"),
           FilterPlayer("filter"),
           StartWordPlayer("slate opener", "slate"),
           StartWordPlayer("fuzzy opener", "fuzzy")]

for player in players:
    print(player.name, average_turns(player, secret_words))

random 7.0
filter 4.24
slate opener 3.8
fuzzy opener 4.68


`FilterPlayer` always opens with `aback`, the first word in `answers.txt`.
`slate` uses common letters, so its first clue rules out many more words.
`fuzzy` uses rare letters and repeats `z`, so its first clue tells the player
very little.  Changing one method of one subclass was enough to try the idea.

## 6. A family of LLM players

The same idea works for LLM players.  `LLMPlayer` splits its work into two
methods: `make_prompt` builds the text sent to the model, and `next_guess`
sends it and checks the reply.  If the reply is not a legal word,
`super().next_guess(history)` falls back to the parent's random guess.
(This needs Ollama running, as in the LLM notebook.)

In [32]:
import ollama

RULES = ("You are playing Wordle. The answer is a 5-letter English word. "
         "After each guess you get feedback for each letter: green means the "
         "right letter in the right spot, yellow means the letter is in the "
         "word but in a different spot, gray means the letter is not in the "
         "word. Reply with only your next guess: one 5-letter word.")

class LLMPlayer(Player):
    def __init__(self, name, model):
        super().__init__(name)
        self.model = model
        self.illegal = 0              # how many times it broke the rules

    def make_prompt(self, history):
        lines = []
        for guess, feedback in history:
            letters = [f"{letter.upper()}={status}" for letter, status in feedback]
            lines.append(guess.upper() + ": " + " ".join(letters))
        if not lines:
            lines = ["(no guesses yet)"]
        return "Guesses so far:\n" + "\n".join(lines)

    def next_guess(self, history):
        messages = [
            {"role": "system", "content": RULES},
            {"role": "user", "content": self.make_prompt(history)},
        ]
        reply = ollama.chat(model=self.model, messages=messages)
        word = reply["message"]["content"].strip().strip(".!").lower()
        if word in legal:
            return word
        self.illegal = self.illegal + 1
        return super().next_guess(history)    # Player's random guess

To change *what the model is told*, a subclass only has to override
`make_prompt`.  This one adds the words that are still possible, so the model
does not have to work out the letter clues by itself:

In [33]:
class HintLLMPlayer(LLMPlayer):
    def make_prompt(self, history):
        prompt = super().make_prompt(history)
        possible = [w for w in answers
                    if all(score(w, g) == fb for g, fb in history)]
        return prompt + "\nWords that are still possible: " + ", ".join(possible[:20])

hint = HintLLMPlayer("llama with hints", "llama3.2")
print(hint.make_prompt([("slate", score("crane", "slate"))]))

Guesses so far:
SLATE: S=gray L=gray A=green T=gray E=green
Words that are still possible: adage, agape, amaze, awake, aware, brace, brake, brave, chafe, crane, crave, craze, drake, drape, evade, frame, grace, grade, grape, grave


Which one does better?  The LLM is slow, so we use only 10 words.

In [34]:
llm = LLMPlayer("llama", "llama3.2")

for player in [llm, hint]:
    print(player.name, average_turns(player, secret_words[:10]),
          "illegal guesses:", player.illegal)

llama 7.0 illegal guesses: 2
llama with hints 5.1 illegal guesses: 5
